# 02 Pembersihan dan Kualitas Data

## Hasil pembelajaran

Setelah notebook ini, Anda dapat:

- Mengidentifikasi nilai hilang dan duplikat;
- Memahami strategi untuk menangani missing values;
- Menghapus duplikat baris;
- Melakukan pembersihan data dasar;
- Menyimpan dataset bersih ke file CSV baru.

### Penanda Sel Kode
- `# RUN-NOW`: dapat langsung dijalankan.
- `# LEARNER-TASK`: ubah bagian bertanda [UBAH]/[ANDA TULIS], lalu jalankan.

## Mengapa data quality penting?

**"Garbage in, garbage out"** adalah prinsip penting dalam data analysis:

- Data buruk → analisis buruk → kesimpulan salah
- Sebelum analisis, **bersihkan data terlebih dahulu**
- Dokumentasikan setiap keputusan pembersihan Anda
- Jangan buang data tanpa alasan yang jelas

## Impor dan muat data

In [10]:
# RUN-NOW: Jalankan sel ini apa adanya.
import pandas as pd
import numpy as np
from pathlib import Path

# Muat data dari notebook sebelumnya (robust terhadap working directory yang berbeda)
candidate_paths = [
    Path("data/synthetic/synthetic_student_performance.csv"),
    Path("../data/synthetic/synthetic_student_performance.csv"),
    Path("/workspaces/metpen-ai-lab/data/synthetic/synthetic_student_performance.csv"),
]

data_file = next((p for p in candidate_paths if p.exists()), None)
if data_file is None:
    raise FileNotFoundError(
        "File data tidak ditemukan. Coba periksa folder data/synthetic atau path notebook."
    )

df = pd.read_csv(data_file)

print(f"Data dimuat dari: {data_file.resolve()}")
print(f"Data asli: {df.shape[0]} baris × {df.shape[1]} kolom")

Data dimuat dari: /workspaces/metpen-ai-lab/data/synthetic/synthetic_student_performance.csv
Data asli: 2012 baris × 20 kolom


## 1. Identifikasi nilai hilang (missing values)

In [11]:
# RUN-NOW: Jalankan sel ini apa adanya.
# Hitung nilai hilang per kolom
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_summary = pd.DataFrame({
    'Kolom': missing_counts.index,
    'Jumlah Hilang': missing_counts.values,
    'Persentase': missing_percent.values
})

missing_summary = missing_summary[missing_summary['Jumlah Hilang'] > 0].sort_values('Jumlah Hilang', ascending=False)

if len(missing_summary) > 0:
    print("Kolom dengan nilai hilang:")
    print(missing_summary)
else:
    print("Tidak ada nilai hilang ditemukan")

Kolom dengan nilai hilang:
                   Kolom  Jumlah Hilang  Persentase
8       final_exam_score             39    1.938370
10  study_hours_per_week              4    0.198807


## 2. Identifikasi duplikat

In [12]:
# RUN-NOW: Jalankan sel ini apa adanya.
# Hitung baris duplikat
num_duplicates = df.duplicated().sum()
print(f"Jumlah baris duplikat: {num_duplicates}")

if num_duplicates > 0:
    print("\nContoh baris duplikat:")
    print(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10))

Jumlah baris duplikat: 12

Contoh baris duplikat:
     student_id  cohort_year     program  semester  attendance_rate  \
535     S100535         2022   Economics         7            90.04   
2009    S100535         2022   Economics         7            90.04   
538     S100538         2021   Education         1            83.96   
2000    S100538         2021   Education         1            83.96   
565     S100565         2023  Psychology         5            86.66   
2002    S100565         2023  Psychology         5            86.66   
627     S100627         2021  Psychology         7            79.99   
2001    S100627         2021  Psychology         7            79.99   
636     S100636         2023   Economics         1            82.68   
2010    S100636         2023   Economics         1            82.68   

      assignment_score  quiz_average  midterm_score  final_exam_score  \
535              87.49         74.75          100.0            100.00   
2009             87.49

## 3. Strategi menangani missing values

Pilihan umum:
- **Hapus baris**: Jika sedikit baris hilang dan tidak mempengaruhi banyak
- **Isi dengan nilai**: Mean, median, atau kategori paling sering
- **Buat flag**: Tandai kolom "missing_reason" untuk analisis selanjutnya
- **Biarkan**: Jika algorithm modern yang Anda gunakan support missing values

**Untuk dataset ini**: Kami akan lihat apakah `missing_reason_flag` column sudah membantu.

In [13]:
# RUN-NOW: Jalankan sel ini apa adanya.
# Periksa missing_reason_flag
if 'missing_reason_flag' in df.columns:
    print("Nilai dalam missing_reason_flag:")
    print(df['missing_reason_flag'].value_counts())
    
    # Periksa korelasi antara flag dan missing values
    print("\nPola missing values:")
    print(df[missing_summary['Kolom'].values].isnull().sum() if len(missing_summary) > 0 else "Tidak ada missing values")

Nilai dalam missing_reason_flag:
missing_reason_flag
none                   1733
self_report_missing     198
system_issue             81
Name: count, dtype: int64

Pola missing values:
final_exam_score        39
study_hours_per_week     4
dtype: int64


## 4. Keputusan pembersihan

In [14]:
# RUN-NOW: Jalankan sel ini apa adanya.
# Buat salinan untuk dibersihkan
df_clean = df.copy()

print("Langkah pembersihan:")

# Hapus duplikat
if num_duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"✓ Hapus {num_duplicates} duplikat")
    print(f"  Dataset sekarang: {df_clean.shape[0]} baris")
else:
    print("✓ Tidak ada duplikat untuk dihapus")

# Untuk missing values numerik
numeric_cols = df_clean.select_dtypes(include=["number"]).columns
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        # Isi dengan median (lebih robust dari mean)
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"✓ Isi missing values di '{col}' dengan median: {median_val:.2f}")

# Untuk missing values kategoris
categorical_cols = df_clean.select_dtypes(include=["object", "string"]).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        # Isi dengan mode (kategori paling sering)
        mode_val = df_clean[col].mode().iloc[0]
        df_clean[col] = df_clean[col].fillna(mode_val)
        print(f"✓ Isi missing values di '{col}' dengan mode: {mode_val}")

print(f"\n✓ Pembersihan selesai")

Langkah pembersihan:
✓ Hapus 12 duplikat
  Dataset sekarang: 2000 baris
✓ Isi missing values di 'final_exam_score' dengan median: 100.00
✓ Isi missing values di 'study_hours_per_week' dengan median: 11.13

✓ Pembersihan selesai


## 5. Verifikasi pembersihan

In [15]:
# RUN-NOW: Jalankan sel ini apa adanya.
# Periksa hasilnya
print("📊 HASIL PEMBERSIHAN")
print(f"Bentuk awal: {df.shape}")
print(f"Bentuk akhir: {df_clean.shape}")
print(f"Baris dihapus: {df.shape[0] - df_clean.shape[0]}")

print(f"\nMissing values sebelum: {df.isnull().sum().sum()}")
print(f"Missing values sesudah: {df_clean.isnull().sum().sum()}")

print(f"\nDuplikat sebelum: {df.duplicated().sum()}")
print(f"Duplikat sesudah: {df_clean.duplicated().sum()}")

📊 HASIL PEMBERSIHAN
Bentuk awal: (2012, 20)
Bentuk akhir: (2000, 20)
Baris dihapus: 12

Missing values sebelum: 43
Missing values sesudah: 0

Duplikat sebelum: 12
Duplikat sesudah: 0


## 6. Periksa outlier atau nilai tidak realistis

In [16]:
# RUN-NOW: Jalankan sel ini apa adanya.
# Periksa distribusi variabel numerik
print("Statistik variabel numerik (data bersih):")
print(df_clean.describe())

# Cek nilai yang mungkin outlier
print("\n⚠️ Periksa: Ada nilai yang terlihat tidak realistis?")
print("Contoh: attendance_rate < 0, final_grade > 100, study_hours > 168/minggu")

Statistik variabel numerik (data bersih):
       cohort_year     semester  attendance_rate  assignment_score  \
count  2000.000000  2000.000000      2000.000000       2000.000000   
mean   2022.498500     4.514500        81.449995         82.428375   
std       1.035149     2.254181        10.024471          9.208494   
min    2021.000000     1.000000        51.200000         56.300000   
25%    2022.000000     3.000000        74.307500         76.032500   
50%    2022.000000     5.000000        81.675000         82.655000   
75%    2023.000000     6.000000        88.610000         88.857500   
max    2024.000000     8.000000       100.000000        100.000000   

       quiz_average  midterm_score  final_exam_score   lms_logins  \
count   2000.000000     2000.00000       2000.000000  2000.000000   
mean      73.920975       98.14814         96.702655    53.635500   
std        8.942234        4.11994          5.709658    11.619214   
min       43.990000       55.21000         61.58000

## 7. Simpan dataset bersih

In [17]:
# RUN-NOW: Jalankan sel ini apa adanya.
# Tentukan folder output
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Simpan dataset bersih
output_file = output_dir / "student_performance_cleaned.csv"
df_clean.to_csv(output_file, index=False)

print(f"✓ Dataset bersih disimpan ke: {output_file}")
print(f"  Ukuran: {output_file.stat().st_size / 1024:.1f} KB")

✓ Dataset bersih disimpan ke: data/processed/student_performance_cleaned.csv
  Ukuran: 226.2 KB


## 8. Dokumentasi keputusan pembersihan

**Catat di sini apa yang Anda lakukan** (penting untuk reproducibility!):

- [ ] Duplikat dihapus: (jumlah) baris
- [ ] Missing values diisi dengan: [median/mode/dropna]
- [ ] Outlier ditangani dengan: [dijelaskan atau dihapus]
- [ ] Total baris hilang: (jumlah)
- [ ] Alasan keputusan: [tulis di sini]

**Contoh dokumentasi untuk Anda:**

In [18]:
# RUN-NOW: Jalankan sel ini apa adanya.
# Tulis ringkasan di sini sebagai dokumentasi
documentation = f"""
DOKUMENTASI PEMBERSIHAN DATA
============================
Dataset: student_performance
Tanggal: 2026-05-09

LANGKAH YANG DILAKUKAN:
1. Duplikat: Hapus {num_duplicates} baris duplikat
   Alasan: Baris identik tidak memberikan informasi tambahan

2. Missing Values: Isi dengan median (numerik) dan mode (kategoris)
   Alasan: Sedikit missing values, isi lebih baik daripada hapus

3. Outlier: [Periksa dengan describe() di atas]
   Alasan: [Keputusan Anda]

HASIL AKHIR:
- Baris: {df.shape[0]} → {df_clean.shape[0]} ({df.shape[0] - df_clean.shape[0]} hilang)
- Kolom: {df.shape[1]} (tidak berubah)
- Missing: {df.isnull().sum().sum()} → {df_clean.isnull().sum().sum()}
- Duplikat: {df.duplicated().sum()} → {df_clean.duplicated().sum()}
"""

print(documentation)


DOKUMENTASI PEMBERSIHAN DATA
Dataset: student_performance
Tanggal: 2026-05-09

LANGKAH YANG DILAKUKAN:
1. Duplikat: Hapus 12 baris duplikat
   Alasan: Baris identik tidak memberikan informasi tambahan

2. Missing Values: Isi dengan median (numerik) dan mode (kategoris)
   Alasan: Sedikit missing values, isi lebih baik daripada hapus

3. Outlier: [Periksa dengan describe() di atas]
   Alasan: [Keputusan Anda]

HASIL AKHIR:
- Baris: 2012 → 2000 (12 hilang)
- Kolom: 20 (tidak berubah)
- Missing: 43 → 0
- Duplikat: 12 → 0



## Langkah berikutnya

✓ Data bersih disimpan di `data/processed/student_performance_cleaned.csv`

→ Notebook berikutnya (`03_exploratory_analysis.ipynb`): Buat visualisasi untuk memahami pola dalam data.